# Análise exploratória da base Olist

Neste notebook você conhece a base do projeto em quatro partes.

1. As oito tabelas, com o significado de cada coluna e a distribuição dos valores.
2. As ligações entre as tabelas e os cuidados de cada junção.
3. A montagem da tabela final de avaliações, passo a passo.
4. As análises que orientaram as decisões do projeto.

Para rodar, abra o notebook a partir da pasta `notebooks/` e deixe os CSVs do Kaggle em `data/raw/`.

## Mapa das tabelas

A tabela de pedidos, `orders`, fica no centro da base. As outras sete tabelas se ligam a ela pelo `order_id` ou por uma tabela intermediária.

![Relação de bases existentes](../images/relacao_base_olist.png)


| Tabela | Uma linha representa | Chave | Usada no projeto |
|---|---|---|---|
| `orders` | um pedido | `order_id` | sim, datas de compra e entrega |
| `order_reviews` | uma avaliação enviada por um cliente | `review_id` | sim, nota e texto |
| `customers` | o cliente de um pedido | `customer_id` | sim, estado |
| `order_items` | um item dentro de um pedido | `order_id` + `order_item_id` | sim, liga pedido e produto |
| `products` | um produto do catálogo | `product_id` | sim, categoria |
| `product_category_name_translation` | uma categoria e seu nome em inglês | `product_category_name` | não |
| `sellers` | um vendedor | `seller_id` | não |
| `order_payments` | um pagamento de um pedido | `order_id` + `payment_sequential` | não |

A base do Kaggle também traz `olist_geolocation_dataset.csv`, com coordenadas por CEP. Deixamos esse arquivo de fora porque a análise dos comentários não usa coordenadas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)

RAW = Path("..") / "data" / "raw"

ARQUIVOS = {
    "orders": "olist_orders_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "translation": "product_category_name_translation.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
}
tabelas = {nome: pd.read_csv(RAW / arquivo) for nome, arquivo in ARQUIVOS.items()}

pd.DataFrame(
    [(nome, arquivo, df.shape[0], df.shape[1]) for (nome, df), arquivo in zip(tabelas.items(), ARQUIVOS.values())],
    columns=["tabela", "arquivo", "linhas", "colunas"],
)

A função abaixo resume qualquer tabela com o tipo de cada coluna, a quantidade de nulos, a quantidade de valores distintos e um exemplo de valor. As seções da Parte 1 usam essa função.

In [ ]:
def resumo_colunas(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "exemplo": df.apply(lambda coluna: coluna.dropna().iloc[0] if coluna.notna().any() else None),
    })

## Parte 1. As tabelas

### 1.1 orders, os pedidos

Uma linha é um pedido. A tabela guarda a situação do pedido e as datas de cada etapa, da compra até a entrega.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `order_id` | Identificador único do pedido | Chave para juntar avaliações, itens e pagamentos |
| `customer_id` | Cliente daquele pedido. Cada pedido recebe um `customer_id` próprio | Junção com `customers` para obter o estado |
| `order_status` | Situação atual do pedido, com valores como `delivered`, `shipped`, `canceled` e `unavailable` | Só na análise |
| `order_purchase_timestamp` | Data e hora da compra | Gera o mês da compra (`purchase_month`) |
| `order_approved_at` | Data e hora da aprovação do pagamento | Não usada |
| `order_delivered_carrier_date` | Data em que o vendedor entregou o pacote à transportadora | Não usada |
| `order_delivered_customer_date` | Data em que o cliente recebeu o pedido | Gera `delivery_status` e `delay_days` |
| `order_estimated_delivery_date` | Prazo de entrega informado ao cliente no momento da compra | Gera `delay_days` |

In [ ]:
orders = tabelas["orders"]
orders.head(3)

In [ ]:
resumo_colunas(orders)

In [ ]:
orders["order_status"].value_counts().to_frame("pedidos")

A célula seguinte converte as datas, que chegam como texto, e lista as situações dos pedidos sem data de entrega ao cliente.

In [ ]:
colunas_data = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
orders[colunas_data] = orders[colunas_data].apply(pd.to_datetime)

sem_entrega = orders["order_delivered_customer_date"].isna()
orders.loc[sem_entrega, "order_status"].value_counts().to_frame("pedidos sem data de entrega")

In [ ]:
dias_entrega = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
prazo_prometido = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days
atraso = (orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"].dt.normalize()).dt.days

print(f"Período das compras: {orders['order_purchase_timestamp'].min():%d/%m/%Y} a {orders['order_purchase_timestamp'].max():%d/%m/%Y}")
print(f"Mediana de dias até a entrega: {dias_entrega.median():.0f}")
print(f"Mediana do prazo prometido em dias: {prazo_prometido.median():.0f}")
print(f"Pedidos entregues com atraso: {(atraso > 0).sum()} ({(atraso > 0).sum() / atraso.notna().sum():.1%} dos entregues)")

In [ ]:
pedidos_por_mes = orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()
ax = pedidos_por_mes.plot(kind="bar", figsize=(12, 3.5), color="#4c72b0")
ax.set_title("Pedidos por mês de compra")
ax.set_xlabel("")
ax.set_ylabel("pedidos")
plt.tight_layout()

Principais achados em `orders`.

- `order_id` e `customer_id` não se repetem. Para contar clientes recorrentes, use `customer_unique_id`, da tabela `customers`.
- A Olist entregou 96,5 mil pedidos. Os 2.965 pedidos sem data de entrega estão em outras situações, como `shipped`, `canceled` e `unavailable`. Oito aparecem como `delivered` sem data, um erro de registro.
- O cliente recebe em 10 dias pela mediana, e o prazo prometido é de 23 dias. Mesmo assim, 6,8% dos pedidos entregues chegaram depois do prazo.
- Os meses de 2016 e os de setembro e outubro de 2018 têm poucos pedidos. Os pedidos se concentram entre janeiro de 2017 e agosto de 2018. Novembro de 2017 tem o pico da Black Friday.

### 1.2 order_reviews, as avaliações

Uma linha é uma resposta à pesquisa de satisfação. Segundo a descrição do dataset no Kaggle, a Olist envia a pesquisa por e-mail quando o cliente recebe o pedido ou quando o prazo estimado vence. Esta é a tabela principal do projeto.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `review_id` | Identificador da avaliação | ID do documento no índice e das citações |
| `order_id` | Pedido avaliado | Junção com `orders` |
| `review_score` | Nota de 1 a 5 dada pelo cliente | Metadado `score` e filtro |
| `review_comment_title` | Título opcional escrito pelo cliente | Parte do texto do documento |
| `review_comment_message` | Comentário opcional escrito pelo cliente | Parte principal do texto do documento |
| `review_creation_date` | Data em que a Olist enviou a pesquisa ao cliente | Não usada |
| `review_answer_timestamp` | Data e hora em que o cliente respondeu | Não usada |

In [ ]:
reviews = tabelas["reviews"]
reviews.head(5)

In [ ]:
resumo_colunas(reviews)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.5))

reviews["review_score"].value_counts().sort_index().plot(kind="bar", ax=eixos[0], color="#4c72b0")
eixos[0].set_title("Avaliações por nota")
eixos[0].set_xlabel("nota")

taxa_comentario = reviews.groupby("review_score")["review_comment_message"].apply(lambda s: s.notna().mean() * 100)
taxa_comentario.plot(kind="bar", ax=eixos[1], color="#dd8452")
eixos[1].set_title("% de avaliações com comentário, por nota")
eixos[1].set_xlabel("nota")
plt.tight_layout()

In [ ]:
taxa_comentario.round(1).to_frame('% com comentário')

A nota 5 aparece em 57,8% das avaliações. Clientes com nota 1 comentam em 76,5% dos casos, contra 35,9% dos clientes com nota 5, e os comentários reúnem mais experiências ruins do que a base. A Parte 4 mede essa distorção.

A célula seguinte compara `review_id` e `order_id`. Numa base sem problemas, cada avaliação teria um ID próprio e cada pedido teria uma avaliação.

In [ ]:
print(f"Linhas: {len(reviews)}")
print(f"review_id distintos: {reviews['review_id'].nunique()}")
print(f"order_id distintos: {reviews['order_id'].nunique()}")

repetidos = reviews[reviews["review_id"].duplicated(keep=False)].sort_values("review_id")
repetidos[["review_id", "order_id", "review_score", "review_comment_message"]].head(4)

O mesmo `review_id` aparece ligado a pedidos diferentes com nota e texto idênticos. Um cliente fez pedidos separados e respondeu uma pesquisa só. A tabela final mantém uma linha por `review_id`.

Alguns pedidos também têm mais de uma avaliação.

In [ ]:
reviews["order_id"].value_counts().value_counts().sort_index().to_frame("pedidos").rename_axis("avaliações por pedido")

As células seguintes medem o tamanho dos textos, listam os mais comuns e contam problemas de formatação.

In [ ]:
comentarios = reviews["review_comment_message"].dropna()
palavras = comentarios.str.split().str.len()

pd.DataFrame({
    "caracteres": comentarios.str.len().describe(),
    "palavras": palavras.describe(),
}).round(1)

In [ ]:
print(f"Comentários com uma palavra: {(palavras == 1).sum()}")
print(f"Comentários com até três palavras: {(palavras <= 3).sum()}")
print(f"Comentários sem nenhuma letra: {(~comentarios.str.contains(r'[A-Za-zÀ-ÿ]')).sum()}")
com_quebra = comentarios.str.contains(r"[\r\n]")
print(f"Comentários com quebra de linha: {com_quebra.sum()}")
print(f"Avaliações só com título: {(reviews['review_comment_title'].notna() & reviews['review_comment_message'].isna()).sum()}")
print(f"Avaliações com título e comentário: {(reviews['review_comment_title'].notna() & reviews['review_comment_message'].notna()).sum()}")

In [ ]:
comentarios.str.strip().str.lower().value_counts().head(15).to_frame("ocorrências")

In [ ]:
comentarios[palavras >= 15].sample(5, random_state=7).to_frame("exemplos de comentários longos")

Principais achados em `order_reviews`.

- 58,7% das avaliações não têm comentário e 88,3% não têm título. As 1.729 avaliações que só têm título ainda carregam informação, como "Não recebi o produto".
- O comentário mais longo tem 208 caracteres e a mediana é de 53. Cada avaliação cabe inteira em um documento, sem divisão em pedaços.
- Os textos mais comuns são elogios curtos, como "muito bom" e "recomendo". Eles dizem pouco sobre o motivo da satisfação.
- 3.852 comentários têm quebras de linha, que precisam virar espaço antes de gerar embeddings.

### 1.3 customers, os clientes

Uma linha é o cliente de um pedido. A tabela tem uma linha para cada `customer_id` de `orders`, sem sobras nem faltas.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `customer_id` | Identificador do cliente naquele pedido | Junção com `orders` |
| `customer_unique_id` | Identificador da pessoa. Se repete quando a mesma pessoa faz vários pedidos | Não usada |
| `customer_zip_code_prefix` | Cinco primeiros dígitos do CEP | Não usada |
| `customer_city` | Cidade do cliente | Não usada |
| `customer_state` | Sigla do estado | Metadado `state` |

In [ ]:
customers = tabelas["customers"]
customers.head(3)

In [ ]:
resumo_colunas(customers)

In [ ]:
pessoas_recorrentes = (customers["customer_unique_id"].value_counts() > 1).sum()
print(f"Pessoas com mais de um pedido: {pessoas_recorrentes}")
customers["customer_state"].value_counts().head(10).to_frame("clientes")

Existem 99.441 `customer_id` e 96.096 pessoas distintas. Só 2.997 pessoas compraram mais de uma vez. São Paulo concentra 42% dos clientes. O projeto usa só o estado, como metadado exibido na tabela de evidências e no contexto enviado ao LLM. Ainda não há filtro por UF nem estatísticas por estado no panorama.

### 1.4 order_items, os itens do pedido

Uma linha é um item dentro de um pedido. Um pedido com duas unidades do mesmo produto tem duas linhas. Esta é a ponte entre pedidos, produtos e vendedores.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `order_id` | Pedido ao qual o item pertence | Junção com `orders` |
| `order_item_id` | Número sequencial do item dentro do pedido, a partir de 1 | Não usada |
| `product_id` | Produto vendido | Junção com `products` para obter a categoria |
| `seller_id` | Vendedor do item | Não usada |
| `shipping_limit_date` | Data limite para o vendedor entregar o item à transportadora | Não usada |
| `price` | Preço do item | Escolhe a categoria principal do pedido |
| `freight_value` | Frete do item. A Olist divide o frete entre os itens de um mesmo pedido | Não usada |

In [ ]:
items = tabelas["items"]
items.head(3)

In [ ]:
resumo_colunas(items)

In [ ]:
items[['order_item_id', 'price', 'freight_value']].describe().round(2)

In [ ]:
items.groupby("order_id").size().value_counts().sort_index().head(8).to_frame("pedidos").rename_axis("itens no pedido")

In [ ]:
itens_com_categoria = items.merge(tabelas["products"][["product_id", "product_category_name"]], on="product_id", how="left")
categorias_por_pedido = itens_com_categoria.groupby("order_id")["product_category_name"].nunique()
print(f"Pedidos com itens: {items['order_id'].nunique()}")
print(f"Pedidos com mais de uma categoria: {(categorias_por_pedido > 1).sum()}")
print(f"Pedidos com mais de um vendedor: {(items.groupby('order_id')['seller_id'].nunique() > 1).sum()}")

A tabela tem 112.650 linhas para 98.666 pedidos, embora 90% dos pedidos tenham um item só. Uma junção de `order_items` com as avaliações, sem agregação, duplicaria avaliações. Por isso a Parte 3 reduz os itens a uma linha por pedido antes da junção. Só 727 pedidos misturam categorias, e neles vale a categoria do item mais caro.

### 1.5 products, os produtos

Uma linha é um produto do catálogo. Os nomes de duas colunas têm o erro de digitação `lenght` no arquivo original.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `product_id` | Identificador do produto | Junção com `order_items` |
| `product_category_name` | Categoria em português, no formato `cama_mesa_banho` | Metadado `category` e filtro |
| `product_name_lenght` | Quantidade de caracteres do nome do produto | Não usada |
| `product_description_lenght` | Quantidade de caracteres da descrição | Não usada |
| `product_photos_qty` | Quantidade de fotos no anúncio | Não usada |
| `product_weight_g` | Peso em gramas | Não usada |
| `product_length_cm` | Comprimento em centímetros | Não usada |
| `product_height_cm` | Altura em centímetros | Não usada |
| `product_width_cm` | Largura em centímetros | Não usada |

A base não traz o nome do produto, só o tamanho do nome. A categoria é a única informação de produto que ajuda a filtrar avaliações.

In [ ]:
products = tabelas["products"]
products.head(3)

In [ ]:
resumo_colunas(products)

In [ ]:
print(f"Categorias distintas: {products['product_category_name'].nunique()}")
products["product_category_name"].value_counts().head(15).to_frame("produtos")

610 produtos não têm categoria, e nesses mesmos produtos faltam nome, descrição e fotos. Na tabela final eles recebem `sem_categoria` para não perder as avaliações.

### 1.6 product_category_name_translation, a tradução das categorias

Uma linha é uma categoria com o nome em inglês.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `product_category_name` | Nome da categoria em português | Não usada |
| `product_category_name_english` | Nome da categoria em inglês | Não usada |

In [ ]:
translation = tabelas["translation"]
sem_traducao = set(products["product_category_name"].dropna()) - set(translation["product_category_name"])
print(f"Categorias na tradução: {len(translation)}")
print(f"Categorias de produtos sem tradução: {sem_traducao}")
translation.head(5)

Os comentários e as perguntas estão em português, então os nomes em português já servem. A tradução também deixa de fora duas categorias usadas em produtos.

### 1.7 sellers, os vendedores

Uma linha é um vendedor do marketplace.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `seller_id` | Identificador do vendedor | Não usada |
| `seller_zip_code_prefix` | Cinco primeiros dígitos do CEP do vendedor | Não usada |
| `seller_city` | Cidade do vendedor | Não usada |
| `seller_state` | Estado do vendedor | Não usada |

In [ ]:
sellers = tabelas["sellers"]
print(resumo_colunas(sellers)[["tipo", "nulos", "valores distintos"]])
sellers["seller_state"].value_counts().head(5).to_frame("vendedores")

São 3.095 vendedores, 60% deles em São Paulo. Deixamos a tabela de fora porque 1.278 pedidos têm mais de um vendedor e a avaliação cobre o pedido inteiro. Nesses pedidos, você não consegue saber a qual vendedor o comentário se refere. Uma evolução possível é filtrar por estado do vendedor em pedidos de vendedor único.

### 1.8 order_payments, os pagamentos

Uma linha é uma forma de pagamento usada em um pedido. Um pedido pago com cartão e voucher tem duas linhas.

| Coluna | Significado | Uso no projeto |
|---|---|---|
| `order_id` | Pedido pago | Não usada |
| `payment_sequential` | Ordem do pagamento quando o pedido usa mais de uma forma | Não usada |
| `payment_type` | `credit_card`, `boleto`, `voucher`, `debit_card` ou `not_defined` | Não usada |
| `payment_installments` | Número de parcelas | Não usada |
| `payment_value` | Valor pago nessa forma de pagamento | Não usada |

In [ ]:
payments = tabelas["payments"]
print(f"Pedidos com mais de uma forma de pagamento: {(payments.groupby('order_id').size() > 1).sum()}")
payments["payment_type"].value_counts().to_frame("pagamentos")

In [ ]:
payments[['payment_sequential', 'payment_installments', 'payment_value']].describe().round(2)

Cartão de crédito responde por 74% dos pagamentos. A forma de pagamento não ajuda a achar avaliações parecidas, então deixamos a tabela de fora. Uma junção sem agregação também duplicaria avaliações, como em `order_items`.

## Parte 2. Ligações entre as tabelas

Antes de juntar tabelas, você precisa conhecer a cardinalidade de cada ligação. Uma junção um para muitos multiplica linhas, e uma chave sem correspondência gera nulos. A célula abaixo testa as ligações usadas no projeto.

In [ ]:
checagens = pd.DataFrame([
    ("order_id é único em orders", orders["order_id"].is_unique),
    ("customer_id é único em customers", customers["customer_id"].is_unique),
    ("product_id é único em products", products["product_id"].is_unique),
    ("todo order_id de reviews existe em orders", reviews["order_id"].isin(orders["order_id"]).all()),
    ("todo customer_id de orders existe em customers", orders["customer_id"].isin(customers["customer_id"]).all()),
    ("todo product_id de items existe em products", items["product_id"].isin(products["product_id"]).all()),
    ("todo pedido tem avaliação", orders["order_id"].isin(reviews["order_id"]).all()),
    ("todo pedido tem item", orders["order_id"].isin(items["order_id"]).all()),
], columns=["checagem", "resultado"])
checagens

In [ ]:
pedidos_sem_item = orders[~orders["order_id"].isin(items["order_id"])]
print(f"Pedidos sem avaliação: {(~orders['order_id'].isin(reviews['order_id'])).sum()}")
print(f"Pedidos sem item: {len(pedidos_sem_item)}")
print(f"Avaliações de pedidos sem item: {reviews['order_id'].isin(pedidos_sem_item['order_id']).sum()}")
pedidos_sem_item["order_status"].value_counts().to_frame("pedidos sem item")

Resumo das ligações usadas:

| Ligação | Cardinalidade | Cuidado na junção |
|---|---|---|
| `reviews` para `orders` por `order_id` | muitos para um | Junção direta. Cada avaliação encontra um pedido, e só um |
| `orders` para `customers` por `customer_id` | um para um | Junção direta |
| `orders` para `order_items` por `order_id` | um para muitos | Reduzir os itens a uma linha por pedido antes de juntar |
| `order_items` para `products` por `product_id` | muitos para um | Junção direta |

Dos 775 pedidos sem item, 767 estão indisponíveis ou cancelados. As avaliações desses pedidos ficam sem categoria na tabela final.

## Parte 3. Montagem da tabela final

A tabela final tem uma linha por avaliação e reúne o texto limpo com os metadados usados nos filtros. Os passos abaixo reproduzem a classe `ReviewDataPreparer` do projeto. No fim, o notebook compara o resultado com a saída da classe.

### Passo 1. Uma linha por avaliação

In [ ]:
base_reviews = reviews.drop_duplicates("review_id").copy()
print(f"Antes: {len(reviews)} linhas. Depois: {len(base_reviews)} linhas.")

### Passo 2. Limpeza do texto

A limpeza troca quebras de linha por espaço, padroniza o Unicode, mascara e-mails, URLs, CPFs e telefones e reduz pontuação repetida. A OpenAI treinou o modelo de embeddings com texto natural, e palavras como "não" mudam o sentido da frase. Por isso a limpeza mantém maiúsculas e palavras.

Depois da limpeza, título e comentário viram um texto só. Se o título já aparece dentro do comentário, `compor_texto` descarta o título.

In [ ]:
import re
import unicodedata

QUEBRAS = re.compile(r"[\r\n\t]+")
ESPACOS = re.compile(r"\s{2,}")
PONTUACAO_REPETIDA = re.compile(r"([!?.,])\1{2,}")
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
URL = re.compile(r"https?://\S+|www\.\S+")
CPF = re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b")
TELEFONE = re.compile(r"\(?\b\d{2}\)?\s?9?\d{4}[-\s]?\d{4}\b")


def limpar_texto(valor):
    if not isinstance(valor, str):
        return None
    texto = unicodedata.normalize("NFC", valor)
    texto = QUEBRAS.sub(" ", texto)
    texto = EMAIL.sub("[email]", texto)
    texto = URL.sub("[url]", texto)
    texto = CPF.sub("[cpf]", texto)
    texto = TELEFONE.sub("[telefone]", texto)
    texto = PONTUACAO_REPETIDA.sub(r"\1", texto)
    texto = ESPACOS.sub(" ", texto).strip()
    return texto or None


def compor_texto(titulo, comentario):
    if titulo and comentario and titulo.lower() in comentario.lower():
        titulo = None
    partes = [parte if parte[-1] in ".!?" else f"{parte}." for parte in (titulo, comentario) if parte]
    return " ".join(partes) if partes else None


base_reviews["text"] = [
    compor_texto(limpar_texto(titulo), limpar_texto(comentario))
    for titulo, comentario in zip(base_reviews["review_comment_title"], base_reviews["review_comment_message"])
]
base_reviews["has_comment"] = base_reviews["review_comment_message"].notna()

In [ ]:
tem_quebra_ou_telefone = base_reviews["review_comment_message"].fillna("").str.contains(r"[\r\n]|\d{4}[-\s]?\d{4}")
base_reviews.loc[tem_quebra_ou_telefone, ["review_comment_title", "review_comment_message", "text"]].head(5)

In [ ]:
base_reviews = base_reviews.rename(columns={"review_score": "score"})[["review_id", "order_id", "score", "text", "has_comment"]]
base_reviews.head(3)

### Passo 3. Informações do pedido

Das datas de `orders` saem três colunas.

- `delay_days` é a diferença em dias entre a entrega e o prazo prometido. Valor negativo significa entrega antes do prazo.
- `delivery_status` vale `nao_entregue` quando não há data de entrega, `atrasado` quando `delay_days` é maior que zero e `no_prazo` nos demais casos.
- `purchase_month` guarda o mês da compra como texto, no formato `2017-11`, e `purchase_month_num` guarda o mesmo mês como o número `201711`. O Chroma aceita maior e menor só em números, e a versão numérica permite filtrar períodos.

O estado do cliente vem de `customers`.

In [ ]:
import numpy as np

base_pedidos = orders.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
entregue = base_pedidos["order_delivered_customer_date"]
compra = base_pedidos["order_purchase_timestamp"]
atraso = (entregue.dt.normalize() - base_pedidos["order_estimated_delivery_date"].dt.normalize()).dt.days

base_pedidos["delay_days"] = atraso.astype("Int64")
base_pedidos["delivery_status"] = np.select(
    [entregue.isna(), atraso > 0], ["nao_entregue", "atrasado"], default="no_prazo"
)
base_pedidos["purchase_month"] = compra.dt.strftime("%Y-%m")
base_pedidos["purchase_month_num"] = compra.dt.year * 100 + compra.dt.month
base_pedidos = base_pedidos.rename(columns={"customer_state": "state"})[
    ["order_id", "purchase_month", "purchase_month_num", "delivery_status", "delay_days", "state"]
]

print(f"Linhas: {len(base_pedidos)} (uma por pedido)")
base_pedidos["delivery_status"].value_counts().to_frame("pedidos")

### Passo 4. Categoria principal de cada pedido

A célula liga os itens aos produtos, ordena do mais caro para o mais barato e mantém a primeira linha de cada pedido. O resultado tem uma linha por pedido.

In [ ]:
itens = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
categoria_principal = (
    itens.sort_values("price", ascending=False)
    .drop_duplicates("order_id")
    .rename(columns={"product_category_name": "category"})[["order_id", "category"]]
)
print(f"Linhas em order_items: {len(items)}. Linhas depois de reduzir: {len(categoria_principal)}")

In [ ]:
exemplo = categorias_por_pedido[categorias_por_pedido > 1].index[0]
print("Itens do pedido:")
print(itens.loc[itens["order_id"] == exemplo, ["order_id", "price", "product_category_name"]].to_string(index=False))
print("\nCategoria escolhida:")
print(categoria_principal.loc[categoria_principal["order_id"] == exemplo].to_string(index=False))

### Passo 5. Junções

As duas junções usam `how="left"` a partir das avaliações e preservam cada avaliação. O `assert` confirma que o número de linhas não mudou.

In [ ]:
base = base_reviews.merge(base_pedidos, on="order_id", how="left")
assert len(base) == len(base_reviews)
print(f"Depois de juntar pedidos: {len(base)} linhas")

base = base.merge(categoria_principal, on="order_id", how="left")
assert len(base) == len(base_reviews)
print(f"Depois de juntar categorias: {len(base)} linhas")

base.isna().sum().to_frame("nulos")

`text` fica nulo nas avaliações sem título e sem comentário. `delay_days` fica nulo nos pedidos não entregues. `category` fica nula nas avaliações de pedidos sem item ou de produtos sem categoria.

### Passo 6. Valores ausentes e marcação dos documentos indexáveis

A categoria ausente vira `sem_categoria`. A coluna `indexable` marca os textos com duas palavras ou mais e ao menos uma letra, que entram no banco vetorial.

In [ ]:
base["category"] = base["category"].fillna("sem_categoria")
texto = base["text"].fillna("")
base["indexable"] = texto.str.contains(r"[A-Za-zÀ-ÿ]") & (texto.str.split().str.len() >= 2)

print(f"Avaliações na tabela final: {len(base)}")
print(f"Com texto: {base['text'].notna().sum()}")
print(f"Indexáveis: {base['indexable'].sum()}")
base.loc[base["text"].notna() & ~base["indexable"], "text"].value_counts().head(10).to_frame("textos que não entram no índice")

### A tabela final

| Coluna | Origem | Cálculo | Uso |
|---|---|---|---|
| `review_id` | `order_reviews` | Direto, sem repetição | ID do documento e das citações |
| `order_id` | `order_reviews` | Direto | Rastreio até o pedido |
| `score` | `order_reviews.review_score` | Direto | Filtro de nota e panorama |
| `text` | `order_reviews` | Título e comentário limpos e unidos | Conteúdo do documento |
| `has_comment` | `order_reviews` | Comentário não nulo | Panorama |
| `purchase_month` | `orders` | Mês da compra em texto | Exibição e contexto do LLM |
| `purchase_month_num` | `orders` | Mês da compra como número `AAAAMM` | Filtro de período |
| `delivery_status` | `orders` | Datas de entrega e prazo | Filtro e panorama |
| `delay_days` | `orders` | Entrega menos prazo, em dias | Metadado |
| `state` | `customers` | Direto | Exibição e contexto do LLM |
| `category` | `order_items` e `products` | Categoria do item mais caro | Filtro de categoria |
| `indexable` | calculada | Duas palavras ou mais, com letras | Separa o que vai ao índice |

O projeto grava a tabela inteira em `reviews_analytics.parquet`, usada no panorama, e só as linhas indexáveis em `reviews_documents.parquet`, usadas no banco vetorial.

In [ ]:
base.sample(5, random_state=3)

In [ ]:
resumo_colunas(base)

### Conferência com o código do projeto

A célula abaixo roda a classe `ReviewDataPreparer` sobre as mesmas tabelas e compara com a tabela montada aqui.

In [ ]:
import sys

sys.path.append(str(Path("..").resolve()))
from voc_rag.config import Settings
from voc_rag.preparation import ReviewDataPreparer

base_projeto = ReviewDataPreparer(Settings()).build(tabelas)
pd.testing.assert_frame_equal(base.reset_index(drop=True), base_projeto.reset_index(drop=True), check_like=True)
print("A tabela do notebook é igual à tabela gerada pelo projeto.")

## Parte 4. Análises que orientaram as decisões

### Situação da entrega e nota

Se atrasos e pedidos não entregues concentram as notas baixas, `delivery_status` serve como filtro.

In [ ]:
nota_por_entrega = pd.crosstab(base["delivery_status"], base["score"], normalize="index").mul(100).round(1)
ax = nota_por_entrega.plot(kind="bar", stacked=True, figsize=(9, 3.5), colormap="RdYlGn")
ax.set_title("Distribuição de notas por situação da entrega (%)")
ax.set_xlabel("")
ax.legend(title="nota", bbox_to_anchor=(1.01, 1))
plt.tight_layout()

In [ ]:
nota_por_entrega

Nota 1 aparece em 6,6% dos pedidos entregues no prazo, em 53,8% dos atrasados e em 70,9% dos não entregues. A distância entre 6,6% e 53,8% justifica o filtro por situação de entrega.

### Período da compra

In [ ]:
por_mes = base[base["purchase_month"].between("2017-01", "2018-08")].groupby("purchase_month").agg(
    avaliacoes=("review_id", "size"),
    pct_nota_1=("score", lambda s: (s == 1).mean() * 100),
    pct_atrasados=("delivery_status", lambda s: (s == "atrasado").mean() * 100),
)
ax = por_mes[["pct_nota_1", "pct_atrasados"]].plot(figsize=(12, 3.5), marker="o")
ax.set_title("% de notas 1 e % de pedidos atrasados por mês de compra")
ax.set_xlabel("")
plt.tight_layout()

In [ ]:
por_mes.loc['2017-09':'2018-04'].round(1)

As curvas de nota 1 e de atraso sobem juntas. Em novembro de 2017, mês da Black Friday, 11,9% das avaliações são de pedidos atrasados, contra 3,9% em outubro. Fevereiro e março de 2018 têm outro pico. Com o filtro por período, você pergunta o que os clientes relataram nesses meses.

### Viés do índice

In [ ]:
comparacao = pd.DataFrame({
    "base completa (%)": base["score"].value_counts(normalize=True).sort_index() * 100,
    "documentos indexados (%)": base.loc[base["indexable"], "score"].value_counts(normalize=True).sort_index() * 100,
}).round(1)
comparacao

A nota 1 é 11,5% da base e 22,0% dos documentos indexados. Clientes insatisfeitos escrevem mais, e os elogios de uma palavra saem do índice. Se o LLM estimasse frequências a partir das avaliações recuperadas, exageraria as reclamações. Por isso o projeto calcula os números do panorama sobre a tabela completa.

## Resumo das decisões

| Achado | Decisão |
|---|---|
| Comentários curtos, no máximo 208 caracteres | Um documento por avaliação, sem chunking |
| 814 `review_id` repetidos com o mesmo texto | Uma linha por `review_id` |
| `order_items` tem várias linhas por pedido | Reduzir a uma categoria por pedido antes de juntar |
| 610 produtos e 775 pedidos sem categoria | Preencher com `sem_categoria` |
| Pedido sem data de entrega | `delivery_status = nao_entregue` |
| Textos de uma palavra e textos sem letras | Fora do índice, dentro da tabela analítica |
| Quebras de linha, dados pessoais e pontuação repetida | Limpeza leve, sem remover palavras |
| Entrega atrasada concentra notas 1 | Metadado e filtro `delivery_status` |
| Nota 1 sobrerrepresentada no índice | Panorama calculado sobre a base completa |
| Vendedores e pagamentos não se ligam ao conteúdo dos comentários | Tabelas fora do projeto |